# Data Ingestion and Cleaning

This notebook handles the ingestion of Open Food Facts data and performs initial cleaning operations using Apache Spark.

## Objectives:
- Load raw Open Food Facts dataset
- Perform data quality assessment
- Clean and standardize the data
- Handle missing values and duplicates
- Save cleaned data for further processing

In [1]:
# Import required libraries
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import requests
import zipfile
import json

# Set up plotting
plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("OpenFoodFactsDataIngestion") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

# Load data (CSV/JSON/parquet)
df = spark.read.parquet('../data/food.parquet')  # Adjust path/type as needed
df.show(5)

Spark Version: 3.5.5
Spark UI: http://Youssef:4040
+-----------+--------------+--------------+------------+-------+--------------------+--------------------+-------------+---------------------+-----------+-------------+--------------------+--------+------------+--------------------+------------------+----------+---------------+------------------------+----------------------+--------------------------+--------------------+--------------------+--------------+--------------+-------------+-------+--------------+---------+--------------------+--------------------+------------+--------------------+--------------------+-------------------------+---------------------------+-------------+-------------------------+----------------------------+--------------------+--------------------+------------------------------------+--------------------------------------+----------------------------------+--------------------------------+--------------------+-------------------+--------------------+---------

In [5]:
df.printSchema()

root
 |-- additives_n: integer (nullable = true)
 |-- additives_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- allergens_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- brands_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- brands: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- categories_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- checkers_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ciqual_food_name_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- cities_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- code: string (nullable = true)
 |-- compared_to_category: string (nullable = true)
 |-- complete: integer (nullable = true)
 |-- completeness: float (nullable = true)
 |-- correctors_tags: array (nullable = true)
 

In [7]:
import os
import requests
from pyspark.sql.functions import col, desc

# Data cleaning: handle missing, normalize units, deduplicate
# ... cleaning code here ...
# Save to MongoDB
# df.write.format('mongo').mode('overwrite').option('uri', 'mongodb://localhost:27017/off.foods').save()

# Download Open Food Facts dataset if not present
data_dir = "../data"
os.makedirs(data_dir, exist_ok=True)

# Check if we have local data or need to download
parquet_file = os.path.join(data_dir, "food.parquet")
csv_file = os.path.join(data_dir, "openfoodfacts.csv")

if not os.path.exists(parquet_file) and not os.path.exists(csv_file):
    print("Downloading Open Food Facts dataset...")
    # Download a subset of the data for development
    url = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv"
    
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(csv_file, 'wb') as file:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    percent = (downloaded / total_size) * 100
                    print(f"\rDownload progress: {percent:.1f}%", end="")
    print("\nDownload completed!")
else:
    print("Data files found locally.")

# Load the dataset
if os.path.exists(parquet_file):
    print("Loading from Parquet file...")
    df_raw = spark.read.parquet(parquet_file)
elif os.path.exists(csv_file):
    print("Loading from CSV file...")
    df_raw = spark.read.option("header", "true") \
                   .option("inferSchema", "true") \
                   .option("multiline", "true") \
                   .option("escape", '"') \
                   .csv(csv_file)
else:
    print("No data file found. Please ensure the dataset is available.")
    raise FileNotFoundError("Dataset not found")

print("\n=== FILTERING BY TARGET COUNTRIES ===")
print(f"Raw dataset loaded: {df_raw.count():,} rows x {len(df_raw.columns)} columns")

Data files found locally.
Loading from Parquet file...
Raw dataset loaded: 3,804,690 rows x 109 columns


In [10]:
# Filter for target countries: Morocco, Spain, and Egypt
print("\n=== FILTERING BY TARGET COUNTRIES ===")
target_countries = ["morocco", "egypt", "spain"]

# Create filter condition for countries
country_filter_condition = None
for country in target_countries:
    # Use array_contains for exact matches or convert array to string for pattern matching
    condition = array_contains(col('countries_tags'), f'en:{country}') | \
                array_contains(col('countries_tags'), country) | \
                array_contains(col('countries_tags'), country.upper()) | \
                array_contains(col('countries_tags'), country.capitalize())
    if country_filter_condition is None:
        country_filter_condition = condition
    else:
        country_filter_condition = country_filter_condition | condition

# Apply country filter
if 'countries_tags' in df_raw.columns:
    df = df_raw.filter(country_filter_condition & col('countries_tags').isNotNull())
    print(f"Filtered dataset: {df.count():,} rows (from Morocco, Spain, and Egypt)")
    
    # Show country distribution
    print("\nCountry distribution in filtered dataset:")
    country_dist = df.groupBy('countries_tags').count().orderBy(desc('count')).limit(20)
    country_dist.show(20, truncate=False)
else:
    print("Warning: 'countries_tags' column not found. Using full dataset.")
    df = df_raw

print(f"\nFinal dataset: {df.count():,} rows x {len(df.columns)} columns")


=== FILTERING BY TARGET COUNTRIES ===
Filtered dataset: 359,679 rows (from Morocco, Spain, and Egypt)

Country distribution in filtered dataset:
+------------------------------------------------------+------+
|countries_tags                                        |count |
+------------------------------------------------------+------+
|[en:spain]                                            |313208|
|[en:morocco]                                          |16620 |
|[en:france, en:spain]                                 |10219 |
|[en:egypt]                                            |2544  |
|[en:france, en:morocco]                               |2091  |
|[en:france, en:italy, en:spain]                       |1586  |
|[en:germany, en:spain]                                |1144  |
|[en:portugal, en:spain]                               |927   |
|[en:italy, en:spain]                                  |791   |
|[en:france, en:germany, en:spain]                     |457   |
|[en:morocco, en:spain

In [11]:
# Data exploration - basic statistics
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.count():,} rows x {len(df.columns)} columns")
print("\n=== COLUMN NAMES ===")
for i, col in enumerate(df.columns[:20]):  # Show first 20 columns
    print(f"{i+1:2d}. {col}")
if len(df.columns) > 20:
    print(f"... and {len(df.columns) - 20} more columns")

# Show sample data
print("\n=== SAMPLE DATA ===")
df.show(5, truncate=False)

=== DATASET OVERVIEW ===
Shape: 359,679 rows x 109 columns

=== COLUMN NAMES ===
 1. additives_n
 2. additives_tags
 3. allergens_tags
 4. brands_tags
 5. brands
 6. categories
 7. categories_tags
 8. checkers_tags
 9. ciqual_food_name_tags
10. cities_tags
11. code
12. compared_to_category
13. complete
14. completeness
15. correctors_tags
16. countries_tags
17. created_t
18. creator
19. data_quality_errors_tags
20. data_quality_info_tags
... and 89 more columns

=== SAMPLE DATA ===
+-----------+-------------------------------------------------------------------------------------------------------------------------------+-------------------------+---------------------------------+-------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------

In [14]:
# Validate country filtering results
print("=== COUNTRY FILTERING VALIDATION ===")

# Check if we have data after filtering
if df.count() == 0:
    print("⚠️  WARNING: No data found for target countries!")
    print("Checking available countries in the dataset...")
    if 'countries_tags' in df_raw.columns:
        all_countries = df_raw.select('countries_tags').filter(col('countries_tags').isNotNull()).distinct().collect()
        print("Sample countries in dataset:")
        for i, row in enumerate(all_countries[:20]):
            print(f"  {i+1}. {row['countries_tags']}")
        
        # Fallback: try broader country matching
        print("\nTrying broader country matching...")
        broader_filter = (
            col('countries_tags').contains('morocco') |
            col('countries_tags').contains('spain') |
            col('countries_tags').contains('egypt') 
        )
        df = df_raw.filter(broader_filter & col('countries_tags').isNotNull())
        print(f"With broader matching: {df.count():,} rows found")
else:
    print(f"✅ Successfully filtered to {df.count():,} products from target countries")
    
    # Analyze the filtered data
    print("\nDetailed country breakdown:")
    from pyspark.sql.functions import col as spark_col, array_contains
    for country in ['spain', 'morocco', 'egypt']:
        # Use array_contains for exact matches and variations
        country_condition = (
            array_contains(spark_col('countries_tags'), f'en:{country}') |
            array_contains(spark_col('countries_tags'), country) |
            array_contains(spark_col('countries_tags'), country.upper()) |
            array_contains(spark_col('countries_tags'), country.capitalize())
        )
        count = df.filter(country_condition).count()
        print(f"  {country}: {count:,} products")
    
    # Sample products from each country
    print("\nSample products by country:")
    sample_by_country = df.select('product_name', 'countries_tags').limit(10)
    sample_by_country.show(10, truncate=False)

# Calculate data reduction
original_count = df_raw.count()
filtered_count = df.count()
reduction_pct = ((original_count - filtered_count) / original_count) * 100

print(f"\n=== DATA REDUCTION SUMMARY ===")
print(f"Original dataset: {original_count:,} products")
print(f"Filtered dataset: {filtered_count:,} products")
print(f"Data reduction: {reduction_pct:.1f}% ({original_count - filtered_count:,} products removed)")
print(f"Processing efficiency: Working with {(filtered_count/original_count)*100:.1f}% of original data")

=== COUNTRY FILTERING VALIDATION ===
✅ Successfully filtered to 359,679 products from target countries

Detailed country breakdown:
  spain: 336,718 products
  morocco: 21,459 products
  egypt: 2,630 products

Sample products by country:
+---------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------+
|product_name                                                                                                                     |countries_tags                                                                              |
+---------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------+
|[{main, Sugar free gum}, {fr, Gum cool watermelon}, {en, Sugar free gum}]             

In [17]:
# Define key columns for our recommendation system
key_columns = [
    "code", "product_name", "generic_name", "brands", "countries_en", 
    "categories_en", "ingredients_text", "allergens_en", "nutriscore_grade", 
    "ecoscore_grade", "nova_group", "energy_100g", "proteins_100g", 
    "carbohydrates_100g", "fat_100g", "fiber_100g", "sugars_100g", 
    "salt_100g", "sodium_100g", "packaging", "image_url", "image_front_url"
]

essential_cols = [
    "product_name","image_url","ingredients_text","categories",
    "brands","nutriscore_score","nutriscore_grade","energy_100g",
    "fat_100g","sugars_100g","proteins_100g","salt_100g"
]
# Check which columns exist in our dataset
available_columns = [col_name for col_name in key_columns if col_name in df.columns]
missing_columns = [col_name for col_name in key_columns if col_name not in df.columns]

print(f"Available key columns ({len(available_columns)}): {available_columns}")
print(f"Missing columns ({len(missing_columns)}): {missing_columns}")

# Select available columns
df_selected = df.select(*available_columns)
print(f"\nSelected dataset shape: {df_selected.count():,} x {len(available_columns)}")

Available key columns (9): ['code', 'product_name', 'generic_name', 'brands', 'ingredients_text', 'nutriscore_grade', 'ecoscore_grade', 'nova_group', 'packaging']
Missing columns (13): ['countries_en', 'categories_en', 'allergens_en', 'energy_100g', 'proteins_100g', 'carbohydrates_100g', 'fat_100g', 'fiber_100g', 'sugars_100g', 'salt_100g', 'sodium_100g', 'image_url', 'image_front_url']

Selected dataset shape: 359,679 x 9


TypeError: 'str' object is not callable

In [19]:
# Import reduce function
from functools import reduce
from pyspark.sql.functions import col

# Filter out records missing essential data
available_essential = [col_name for col_name in essential_cols if col_name in available_columns]
if available_essential:
    initial_count = df_selected.count()
    df_selected = df_selected.filter(
        reduce(lambda a, b: a & b, [col(col_name).isNotNull() for col_name in available_essential])
    )
    filtered_count = df_selected.count()
    print(f"Records with essential data: {filtered_count:,} (removed {initial_count - filtered_count:,})")

Records with essential data: 215,868 (removed 143,811)


In [23]:
# Data Quality Assessment
print("=== DATA QUALITY ASSESSMENT ===")

# Import col function to avoid conflicts
from pyspark.sql.functions import col as spark_col

# Missing values analysis
missing_stats = []
total_rows = df_selected.count()

for column in df_selected.columns:
    # Handle complex data types (arrays, structs) differently
    if column in ['product_name', 'generic_name', 'ingredients_text']:
        # For array columns, check if null or empty array
        missing_count = df_selected.filter(
            spark_col(column).isNull() | 
            (size(spark_col(column)) == 0)
        ).count()
    else:
        # For simple data types
        missing_count = df_selected.filter(
            spark_col(column).isNull() | 
            (spark_col(column) == "")
        ).count()
    
    missing_pct = (missing_count / total_rows) * 100
    missing_stats.append((column, missing_count, missing_pct))

# Sort by missing percentage
missing_stats.sort(key=lambda x: x[2], reverse=True)

print("\nMissing Values Analysis:")
print(f"{'Column':<25} {'Missing Count':<15} {'Missing %':<10}")
print("-" * 50)
for column_name, count, pct in missing_stats:
    print(f"{column_name:<25} {count:<15,} {pct:<10.2f}%")

=== DATA QUALITY ASSESSMENT ===

Missing Values Analysis:
Column                    Missing Count   Missing % 
--------------------------------------------------
ecoscore_grade            211,614         98.03     %
generic_name              198,473         91.94     %
packaging                 193,388         89.59     %
nova_group                168,984         78.28     %
ingredients_text          167,599         77.64     %
brands                    14,612          6.77      %
product_name              9,481           4.39      %
code                      0               0.00      %
nutriscore_grade          0               0.00      %


In [ ]:
# Data Cleaning Operations
print("=== DATA CLEANING ===")

# 1. Remove duplicates based on product code
initial_count = df_selected.count()
if 'code' in df_selected.columns:
    df_cleaned = df_selected.dropDuplicates(['code'])
    duplicates_removed = initial_count - df_cleaned.count()
    print(f"Duplicates removed: {duplicates_removed:,}")
else:
    df_cleaned = df_selected
    print("No code column found for duplicate removal")

# 2. Filter out products with no product name
if 'product_name' in df_cleaned.columns:
    before_name_filter = df_cleaned.count()
    df_cleaned = df_cleaned.filter(
        col('product_name').isNotNull() & 
        (col('product_name') != "") & 
        (length(col('product_name')) > 2)
    )
    after_name_filter = df_cleaned.count()
    name_filtered = before_name_filter - after_name_filter
    print(f"Products without valid names removed: {name_filtered:,}")

# 3. Clean and standardize text columns
text_columns = ['product_name', 'generic_name', 'brands', 'categories_en', 
                'ingredients_text', 'allergens_en']

for col_name in text_columns:
    if col_name in df_cleaned.columns:
        df_cleaned = df_cleaned.withColumn(
            col_name,
            when(col(col_name).isNull() | (col(col_name) == ""), None)
            .otherwise(trim(lower(col(col_name))))
        )

print(f"Text columns cleaned: {[c for c in text_columns if c in df_cleaned.columns]}")

=== DATA CLEANING ===


In [ ]:
# 4. Handle nutritional values - convert to numeric and handle outliers
nutrition_columns = ['energy_100g', 'proteins_100g', 'carbohydrates_100g', 
                     'fat_100g', 'fiber_100g', 'sugars_100g', 'salt_100g', 'sodium_100g']

for col_name in nutrition_columns:
    if col_name in df_cleaned.columns:
        # Convert to double and handle negative values
        df_cleaned = df_cleaned.withColumn(
            col_name,
            when(col(col_name).cast("double") < 0, None)
            .otherwise(col(col_name).cast("double"))
        )

print(f"Nutrition columns processed: {[c for c in nutrition_columns if c in df_cleaned.columns]}")

# 5. Standardize categorical columns
if 'nutriscore_grade' in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn(
        'nutriscore_grade',
        upper(col('nutriscore_grade'))
    )

if 'ecoscore_grade' in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn(
        'ecoscore_grade',
        upper(col('ecoscore_grade'))
    )

print("\n=== CLEANING SUMMARY ===")
print(f"Original records: {initial_count:,}")
print(f"Final records: {df_cleaned.count():,}")
print(f"Records removed: {initial_count - df_cleaned.count():,}")
print(f"Data retention: {(df_cleaned.count() / initial_count) * 100:.2f}%")

In [ ]:
# Add derived columns for better analysis
print("=== ADDING DERIVED COLUMNS ===")

# 1. Add completeness score (percentage of non-null values)
completeness_cols = [c for c in df_cleaned.columns if c in available_columns]
completeness_expr = sum([when(col(c).isNotNull(), 1).otherwise(0) for c in completeness_cols])
df_cleaned = df_cleaned.withColumn(
    'completeness_score',
    (completeness_expr / len(completeness_cols) * 100).cast('int')
)

# 2. Add processing timestamp
df_cleaned = df_cleaned.withColumn(
    'processed_at',
    current_timestamp()
)

# 3. Extract country information
if 'countries_en' in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn(
        'primary_country',
        when(col('countries_en').isNotNull(),
             split(col('countries_en'), ',')[0])
        .otherwise(None)
    )

print("Derived columns added successfully!")
print(f"Final dataset shape: {df_cleaned.count():,} x {len(df_cleaned.columns)}")

In [ ]:
# Save cleaned data with country filtering information
output_path = "../data/cleaned_food_data_filtered.parquet"

print(f"Saving cleaned and filtered data to {output_path}...")
print(f"Dataset contains products from: Morocco, Spain, and Egypt")

# Write as parquet with partitioning for better performance
df_cleaned.coalesce(5).write \
    .mode('overwrite') \
    .option('compression', 'snappy') \
    .parquet(output_path)

print("Data saved successfully!")

# Also save a sample for quick testing
sample_path = "../data/sample_food_data_filtered.parquet"
df_sample = df_cleaned.sample(0.2)  # 20% sample since we have less data
df_sample.coalesce(1).write \
    .mode('overwrite') \
    .parquet(sample_path)

print(f"Sample data (20%) saved to {sample_path}")
print(f"Sample size: {df_sample.count():,} records")

# Save filtering metadata
filtering_metadata = {
    'filtered_countries': ['Morocco', 'Spain', 'Egypt'],
    'total_products_after_filtering': df_cleaned.count(),
    'filtering_date': datetime.now().isoformat(),
    'data_source': 'Open Food Facts',
    'note': 'Dataset filtered to include only products from Morocco, Spain, and Egypt'
}

import json
with open('../data/filtering_metadata.json', 'w') as f:
    json.dump(filtering_metadata, f, indent=2)

print("\nFiltering metadata saved to: ../data/filtering_metadata.json")

In [ ]:
# Final data summary for filtered dataset
print("=== FINAL DATA SUMMARY (FILTERED) ===")
print(f"Target Countries: Morocco, Spain, and Egypt")
print(f"Total records: {df_cleaned.count():,}")
print(f"Total columns: {len(df_cleaned.columns)}")

# Show country distribution in final dataset
if 'primary_country' in df_cleaned.columns:
    print("\nCountry distribution in cleaned dataset:")
    country_summary = df_cleaned.groupBy('primary_country').count().orderBy(desc('count'))
    country_summary.show(10, truncate=False)

# Show sample of cleaned data with focus on target countries
print("\nSample of cleaned data from target countries:")
df_cleaned.select('product_name', 'brands', 'categories_en', 'nutriscore_grade', 
                  'primary_country', 'completeness_score').show(10, truncate=False)

# Summary statistics
print("\n=== PROCESSING SUMMARY ===")
print(f"✅ Successfully processed products from Morocco, Spain, and Egypt")
print(f"✅ Data cleaning and standardization completed")
print(f"✅ Derived features added (completeness score, primary country)")
print(f"✅ Optimized dataset size for efficient model training")
print(f"✅ Ready for feature engineering and recommendation system development")

# Stop Spark session
spark.stop()
print("\n✅ Spark session stopped. Data ingestion and cleaning completed successfully!")
print("\n📁 Next steps:")
print("   1. Run feature_engineering_eda.ipynb for feature creation")
print("   2. Run recommender_training_evaluation.ipynb for model training")
print("   3. Dataset is now optimized for your target markets")